<table style="width: 100%; border-collapse: collapse; border: none; background: #fffbeb; border-left: 6px solid #f59e0b; border-radius: 8px; padding: 20px; box-shadow: 0 2px 4px rgba(0,0,0,0.05);">
  <tr style="border: none;">
    <td style="vertical-align: middle; border: none; padding: 15px 20px;">
      <h1 style="margin: 0; color: #78350f; font-size: 2em; font-family: system-ui, -apple-system, sans-serif; font-weight: 800; letter-spacing: -0.02em;">
        💡 La Tienda de Doña Rosa: Quién Toca la Puerta, Quién Tiene la Llave y Quién Reconoce tu Cara 🔑
      </h1>
      <p style="margin: 6px 0 0 0; color: #b45309; font-size: 1.15em; font-weight: 600; font-family: system-ui, -apple-system, sans-serif;">
        Especialización en Ciencia de Datos | Big Data
      </p>
      <p style="margin: 4px 0 0 0; color: #92400e; font-size: 0.95em; font-family: system-ui, -apple-system, sans-serif;">
        Universidad Santo Tomás — Seccional Tunja
      </p>
    </td>
    <td style="text-align: right; vertical-align: middle; border: none; padding: 15px 20px; width: 30%;">
      <span style="background: #f59e0b; color: #ffffff; padding: 6px 14px; border-radius: 20px; font-size: 0.85em; font-weight: 700; display: inline-block; margin-bottom: 8px;">
        💡 Para Dummies • Módulo 04
      </span><br>
      <span style="color: #78350f; font-size: 0.85em;">Docente: Santiago A. Zúñiga M.</span><br>
      <a href="mailto:gestorvirtualcienciadatos@ustatunja.edu.co" style="color: #b45309; font-size: 0.8em; text-decoration: none; font-weight: 500;">gestorvirtualcienciadatos@ustatunja.edu.co</a>
    </td>
  </tr>
</table>

<div align="center" style="margin-top: 15px; margin-bottom: 15px;">
  <a href="https://colab.research.google.com/github/sazuniga06/Data-Science-Programming---USTA-Tunja-Repository/blob/main/Big%20Data/04%20-%20Seguridad%20y%20Gobernanza%20de%20Big%20Data/Para%20Dummies/00_Amenazas_Seguridad_Dummies.ipynb" target="_parent">
    <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab" style="vertical-align: middle;"/>
  </a>
</div>

---
## ¿Qué vamos a aprender aquí? 🎈

Este cuaderno es la versión **«para no ingenieros»** del [cuaderno 00 del Módulo 04 (Amenazas de seguridad)](../00_Amenazas_de_Seguridad.ipynb). Doña Rosa tiene una **tienda en Villa de Leyva** y guarda en un cuaderno **quién le debe plata**. Ese cuaderno es un tesoro: ¿qué le puede pasar?

| En la tienda de Doña Rosa | En seguridad de datos |
|---|---|
| Que **alguien ajeno lo lea** (el vecino chismoso) | Se rompe la **confidencialidad** |
| Que **alguien cambie un número** del fiado | Se rompe la **integridad** |
| Que **la tienda esté cerrada** cuando la gente la necesita | Se rompe la **disponibilidad** |
| El que **golpea la puerta 30 veces** probando llaves | **Fuerza bruta** |
| El empleado que **abre la bodega de la plata** sin permiso | **Acceso no autorizado** |
| «Le quité el nombre, es anónimo»… pero **todos reconocen** a la señora de 78 años de la vereda | **Reidentificación** (fuga de datos) |

*(Todos los datos de este cuaderno son **inventados**, y las "amenazas" son juegos con tablas pequeñas del propio cuaderno.)* Al terminar podrás explicar:
1. Qué son la **confidencialidad, la integridad y la disponibilidad** y cómo se ve **un intento de entrar a la fuerza** en un cuaderno de registro.
2. Qué es una **tabla de quién tiene qué llave** y cómo descubrir quién abrió lo que no debía.
3. Por qué **quitar el nombre no basta** y cómo contar cuántos vecinos se parecen a ti.

> 🔗 Si luego quieres ver la versión **con una bitácora grande, un modelo que detecta lo raro, texto con datos personales y reidentificación a escala**: [cuaderno estándar](../00_Amenazas_de_Seguridad.ipynb). Después de esto: [Candados, huellas y libros a prueba de borrones](01_Tecnicas_Seguridad_Dummies.ipynb).

---
## 1. El Cuaderno de la Portería: ¿Quién Golpea la Puerta de Más? 🚪

En el edificio donde vive Doña Rosa, el portero anota **cada intento de entrar al cuarto de los contadores** con la **hora**, **quién** lo intentó y **si falló**. Una noche, entre los vecinos normales, alguien **prueba llave tras llave**. Contemos cuántos **intentos fallidos** hizo cada persona **en una misma ventana de 5 minutos** y levantemos una alarma si pasa de **5** (datos **inventados**).

In [ ]:
import pandas as pd

# Cuaderno del portero (inventado): hora en minutos desde las 22:00, quien lo intento y si fallo
registro = pd.DataFrame(
    [(0, "Marta", True), (3, "Pedro", True), (9, "Luz", True), (14, "Pedro", False)]            # intentos normales de vecinos
    + [(20 + k, "desconocido", True) for k in range(0, 8)]                                       # 8 fallos del desconocido en 8 minutos (probando llaves)
    + [(21, "Marta", True), (21, "Marta", False)],                                               # Marta se equivoca una vez
    columns=["minuto", "persona", "fallo"])
registro["fallo"] = registro["fallo"].astype(bool)
registro = registro.sort_values("minuto").reset_index(drop=True)

fallos = registro[registro.fallo]
def max_fallos_en_5_minutos(minutos):
    """Para cada intento, cuantos fallos de esa persona hubo en los 5 minutos que terminan ahi; devuelve el mayor."""
    m = sorted(minutos)
    return max(sum(1 for t in m if x - 5 < t <= x) for x in m)

resumen = fallos.groupby("persona").minuto.apply(max_fallos_en_5_minutos).rename("máx. fallos en 5 min")
alarma = resumen[resumen >= 5]
print(resumen.to_string())
print("\n¡Alarma! Personas con 5 o más fallos en 5 minutos:", list(alarma.index))
assert list(alarma.index) == ["desconocido"] and resumen["Marta"] == 1 and resumen["Pedro"] == 1

---
**Lección:** el desconocido hizo **8 fallos seguidos** (hasta **5 dentro de una misma ventana de 5 minutos**), mientras que Marta y Pedro se equivocaron **una vez**: equivocarse es humano, **probar sin parar** no. La regla del portero es simple y **explicable**: *«más de 5 fallos en 5 minutos = alarma»*. El **truco** está en elegir bien el número: si lo pones **muy bajo** (2 fallos), la alarma suena por cada vecino distraído (**falsas alarmas**); si lo pones **muy alto** (50), el ladrón paciente que intenta **49 veces** pasa **sin que suene nada**. Y si **una familia entera** olvida las llaves a las 6 de la tarde, sonará la alarma por «el edificio», aunque **cada persona** falló una sola vez: por eso se cuenta **por persona**, no por puerta. *(En el [cuaderno estándar](../00_Amenazas_de_Seguridad.ipynb) esto se mide con una bitácora de 50 000 eventos, y con un modelo que detecta comportamientos raros que no están en ninguna regla.)*

---

## 2. ¿Quién Tiene Qué Llave? La Tabla de Permisos 🗝️

En la tienda hay tres cuartos: la **caja** (la plata), la **bodega** (la mercancía) y la **oficina** (los papeles). **No todos** deben entrar a todo: la regla de oro es **«cada quien solo la llave que necesita»** (*privilegio mínimo*). Escribimos una **tabla de permisos** y revisamos el **cuaderno de entradas** para ver quién abrió algo que **no le tocaba**.

In [ ]:
# Tabla de permisos: quien (por cargo) puede entrar a que cuarto
permisos = {"Doña Rosa": {"caja", "bodega", "oficina"}, "Pedro (celador)": {"bodega"}, "Luz (ayudante)": {"bodega", "oficina"}}

# Cuaderno de entradas de la semana (inventado)
entradas = [("Doña Rosa", "caja"), ("Pedro (celador)", "bodega"), ("Luz (ayudante)", "oficina"), ("Pedro (celador)", "caja"),
            ("Luz (ayudante)", "bodega"), ("Pedro (celador)", "oficina"), ("Doña Rosa", "bodega")]

sin_permiso = [(quien, cuarto) for quien, cuarto in entradas if cuarto not in permisos[quien]]
print("Entradas revisadas:", len(entradas))
print("Entradas SIN permiso:", sin_permiso)
assert sin_permiso == [("Pedro (celador)", "caja"), ("Pedro (celador)", "oficina")]

# ¿y si el problema NO es la llave sino lo que se hace con ella? (Luz tiene permiso de bodega, pero ¿cuántas veces entra?)
visitas = pd.Series(["Luz (ayudante)"] * 2 + ["Pedro (celador)"] * 3 + ["Doña Rosa"] * 2).value_counts()
print("\nVisitas por persona:", visitas.to_dict())

---
**Lección:** con la tabla en la mano, encontrar **quién abrió lo que no debía** es solo **comparar cada entrada con la tabla**: Pedro entró a la **caja** y a la **oficina** y **no tenía permiso** para ninguno de los dos. Lo difícil no es la comparación; lo difícil es **tener la tabla al día** (en las empresas grandes los permisos **se acumulan**: alguien cambia de cargo y conserva las llaves viejas) y **que el portero anote todo** (si no se anota, no se puede revisar). Una advertencia: la tabla **no** detecta al empleado que **sí tiene** permiso pero lo **usa de más** (el ayudante que se lleva mercancía poco a poco): para eso hace falta vigilar **cuánto** y **a qué hora**, no solo **si** tiene la llave.

---

## 3. «Le Quité el Nombre»: ¿Cuántos Vecinos se Parecen a Ti? 🕵️

El centro de salud publica una lista **sin nombres** de personas con una enfermedad, para un estudio. Pero deja la **vereda**, el **sexo** y la **edad exacta**. En el pueblo **todos se conocen**: si en la lista aparece **una mujer de 78 años de la vereda El Salitre** y solo hay **una** así en la vereda, **ya sabes quién es** y qué enfermedad tiene. Medimos **cuántas personas comparten cada combinación** (si da 1, esa persona se **reconoce**) y luego probamos **agrupar edades por rangos**.

In [ ]:
# Lista "anonima" del centro de salud (inventada): vereda, sexo, edad y enfermedad (lo sensible)
lista = pd.DataFrame({
    "vereda": ["El Salitre", "El Salitre", "El Salitre", "El Salitre", "San Isidro", "San Isidro", "San Isidro", "San Isidro", "San Isidro", "Monquirá", "Monquirá", "Monquirá"],
    "sexo":   ["F", "F", "M", "M", "F", "F", "F", "M", "M", "F", "M", "M"],
    "edad":   [78, 31, 45, 47, 29, 33, 36, 52, 58, 64, 41, 44],
    "enfermedad": ["diabetes", "asma", "asma", "asma", "diabetes", "asma", "asma", "diabetes", "asma", "asma", "diabetes", "asma"]})

def cuantos_se_parecen(df, columnas):
    """Tamano de cada grupo de personas con los mismos datos; el MINIMO nos dice cuan protegida esta la persona mas expuesta."""
    return df.groupby(columnas, observed=True).size()

exacto = cuantos_se_parecen(lista, ["vereda", "sexo", "edad"])
print("Con la edad EXACTA, grupos de tamaño 1 (personas que se reconocen):", int((exacto == 1).sum()), "de", len(lista), "| el grupo más chico tiene", int(exacto.min()), "persona")

lista["rango_edad"] = pd.cut(lista.edad, bins=[0, 39, 59, 120], labels=["menos de 40", "40 a 59", "60 o más"])
por_rango = cuantos_se_parecen(lista, ["vereda", "sexo", "rango_edad"])
print("Con la edad en RANGOS, grupos de tamaño 1:", int((por_rango == 1).sum()), "| el grupo más chico tiene", int(por_rango.min()), "persona(s)")
assert int((exacto == 1).sum()) == 12 and int(exacto.min()) == 1
assert int(por_rango.min()) == 1 and int((por_rango == 1).sum()) == 3
print("\nLa señora de 78 años de El Salitre:", lista[lista.edad == 78][["vereda", "sexo", "rango_edad", "enfermedad"]].to_dict("records")[0], "-> sigue siendo la única de su grupo")

---
**Lección:** con la edad **exacta**, **las 12** personas son **únicas** en su combinación (vereda + sexo + edad): cualquiera que las conozca las reconoce. Al agrupar la edad en **rangos**, los grupos **crecen** y las personas únicas **bajan de 12 a 3**, pero **no desaparecen**: la señora de 78 años sigue siendo la **única** mujer de **60 o más** en El Salitre. La solución honesta es **agrupar más** (por ejemplo, la vereda por **zona**) o **no publicar** ese dato: **cuanto más agrupas, más proteges y menos detalle te queda para el estudio**. *(En el [cuaderno estándar](../00_Amenazas_de_Seguridad.ipynb) se hace con **10 000 registros** y un padrón público: sin agrupar, **más del 90 %** se reconoce al enlazar las dos listas.)*

---

## Resumen en una frase 🎯

> **Proteger datos es cuidar tres cosas (que nadie ajeno los vea, que nadie los cambie y que estén cuando se necesiten); en la práctica se vigila quién golpea la puerta de más, se revisa quién abre lo que no debe con una tabla de permisos, y se recuerda que «quitar el nombre» no protege si los demás datos te delatan.**

### Lo que aprendiste hoy:
- ✅ **Confidencialidad, integridad y disponibilidad:** lo que hay que cuidar de los datos.
- ✅ **Fuerza bruta:** *más de 5 fallos en 5 minutos* por persona suena la alarma; un umbral muy bajo da falsas alarmas y uno muy alto deja pasar al paciente.
- ✅ **Tabla de permisos:** revisar entradas contra la tabla encuentra a quien abrió lo que no debía; lo difícil es mantenerla al día.
- ✅ **Reidentificación:** con la edad exacta, las 12 personas se reconocen; en rangos quedan 3; agrupar ayuda, pero cuesta detalle.

> 🎓 **Siguiente paso:** el [cuaderno estándar](../00_Amenazas_de_Seguridad.ipynb) lo hace **a escala**, con modelos y con texto. Para seguir: [Candados, huellas y libros a prueba de borrones](01_Tecnicas_Seguridad_Dummies.ipynb).

---
##### 🛠️ Práctica 1: ¿Cuántos vecinos se parecen a ti? Por zona en vez de por vereda

Usa la tabla `lista` y la función `cuantos_se_parecen` de arriba:

1. Crea la columna `lista["zona"]`: `"Centro"` para las veredas `"El Salitre"` y `"San Isidro"`, y `"Norte"` para `"Monquirá"` (sugerencia: `lista["vereda"].map({...})`).
2. Calcula `por_zona = cuantos_se_parecen(lista, ["zona", "sexo", "rango_edad"])`.
3. Comprueba con `assert` que el grupo **más chico** de `por_zona` es **mayor o igual** que el de `por_rango` y cuenta con `solos_zona` cuántas personas siguen siendo únicas (grupo de tamaño 1): debe ser **menor** que las de `por_rango` (3).

In [ ]:
# Escribe tu código aquí

# lista["zona"] = lista["vereda"].map({...})
# por_zona = cuantos_se_parecen(lista, [...])
# solos_zona = ...


<details>
<summary><b>💡 Haz clic aquí para ver la solución guiada...</b></summary>

```python
lista["zona"] = lista["vereda"].map({"El Salitre": "Centro", "San Isidro": "Centro", "Monquirá": "Norte"})
por_zona = cuantos_se_parecen(lista, ["zona", "sexo", "rango_edad"])
solos_zona = int((por_zona == 1).sum())
print("Grupo más chico por zona:", int(por_zona.min()), "| personas únicas por zona:", solos_zona, "| por vereda:", int((por_rango == 1).sum()))
assert por_zona.min() >= por_rango.min() and solos_zona < int((por_rango == 1).sum())
print("Agrupar la vereda en zonas protege más, pero ahora el estudio ya no sabe en qué vereda vive cada persona: protege más, informa menos.")
```
</details>

---


---
### 🧠 ¿Ya lo tienes claro?

1. Con tus palabras: ¿qué diferencia hay entre que alguien **lea** el cuaderno de fiados y que alguien **cambie** un número? ¿Cuál de las dos es peor para Doña Rosa y por qué?
2. Si el portero pone la alarma en «2 fallos en 5 minutos», ¿qué problema aparece? ¿Y si la pone en «50»? ¿Qué número elegirías para tu edificio y por qué?
3. Una lista dice: «mujer, 78 años, El Salitre, diabetes», sin nombre. ¿Por qué todos en el pueblo saben quién es? ¿Qué le quitarías o cambiarías a la lista para protegerla?

---
<div align="center">
  <p style="font-size: 0.9em; color: #78350f;">
    © 2026 <b>Universidad Santo Tomás — Seccional Tunja</b><br>
    <i>Especialización en Ciencia de Datos | Big Data (Edición Didáctica Para Dummies)</i>
  </p>
</div>